In [42]:
import joblib
data = joblib.load("D:\\DEPI\\artifacts\\scientific.joblib")
print(data.keys())

dict_keys(['detect_target', 'clean_column_names', 'drop_high_nulls', 'feature_engineering', 'impute_missing', 'encoding', 'outlier_handling', 'scaling', 'final_features'])


In [43]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field
import pandas as pd
import joblib
import os
import numpy as np
import uuid
from data_preprocessor2 import preprocess_exoplanet_data

app = FastAPI()

def smart_read(path: str) -> pd.DataFrame:
    """Read a CSV or Excel file and automatically detect headers."""
    if path.lower().endswith(".csv"):
        return pd.read_csv(path, comment="#", engine="python")
    elif path.lower().endswith((".xls", ".xlsx")):
        raw = pd.read_excel(path, header=None, dtype=str)
        header_row = None

        for i, row in raw.iterrows():
            if row.dropna().str.contains(r"(pl_|koi_|st_)", case=False, regex=True).any():
                header_row = i
                break

        if header_row is None:
            raise ValueError("❌ Could not detect a valid header row in Excel file.")

        df = pd.read_excel(path, skiprows=header_row)
        return df
    else:
        raise ValueError(f"❌ Unsupported file type: {path}")


BASE_OUTPUT_DIR = "predictions"
AMATEUR_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "amateur")
SCIENTIFIC_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "scientific")

os.makedirs(AMATEUR_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCIENTIFIC_OUTPUT_DIR, exist_ok=True)


#amature artifacts
MODEL_PATH = r"D:\DEPI\artifacts\amature.joblib"

try:
    saved = joblib.load(MODEL_PATH)
    model = saved["model"]
    scaler = saved["scaler"]
    features = saved["features"]
    defaults = saved.get("defaults", {})
    fill_values = saved.get("fill_values", {})
    feature_order = saved.get("feature_order", list(features.values()))
except Exception as e:
    raise RuntimeError(f"Failed to load amateur model artifacts: {e}")


#scientific artifacts
SCIENTIFIC_PATH = r"D:\DEPI\artifacts\scientific.joblib"

try:
    scientific_saved = joblib.load(SCIENTIFIC_PATH)
    detect_target = scientific_saved["detect_target"]
    clean_column_names = scientific_saved["clean_column_names"]
    drop_high_nulls = scientific_saved["drop_high_nulls"]
    feature_engineering = scientific_saved["feature_engineering"]
    impute_missing = scientific_saved["impute_missing"]
    encoding = scientific_saved["encoding"]
    outlier_handling = scientific_saved["outlier_handling"]
    scaling = scientific_saved["scaling"]
    final_features = scientific_saved["final_features"]
except Exception as e:
    raise RuntimeError(f"❌ Failed to load scientific artifacts: {e}")


class ExoplanetInput(BaseModel):
    transit_depth: float = Field(..., gt=0)
    transit_duration: float = Field(..., gt=0)
    transit_epoch: float = Field(..., gt=0)
    orbital_period: float = Field(..., gt=0)
    planet_to_star_radius_ratio: float = Field(..., gt=0)
    planet_radius: float = Field(..., gt=0)
    stellar_radius: float = Field(..., gt=0)
    temperature: float = Field(..., gt=0)
    stellar_temperature: float = Field(..., gt=0)
    distance: float = Field(..., gt=0)


#amature predict
@app.post("/predict-single/")
def predict_single(input: ExoplanetInput):
    try:
        input_dict = input.dict()

        # Mapping
        mapped_input = {}
        for user_feat, dataset_feat in features.items():
            mapped_input[dataset_feat] = input_dict.get(
                user_feat, defaults.get(user_feat, 0)
            )

        df_input = pd.DataFrame([mapped_input])

        # Preprocessing
        for col, median_val in fill_values.items():
            if col in df_input.columns:
                df_input[col] = pd.to_numeric(df_input[col], errors="coerce").fillna(median_val)

        # ✅ FIX #6: Ensure same number/order of features as scaler expects
        df_input = df_input.reindex(columns=feature_order, fill_value=0)
        if hasattr(scaler, "n_features_in_"):
            expected_features = scaler.n_features_in_
            if df_input.shape[1] != expected_features:
                raise HTTPException(
                    status_code=400,
                    detail=f"Feature mismatch: model expects {expected_features}, got {df_input.shape[1]}."
                )

        X_scaled = scaler.transform(df_input)

        #Model
        probs = model.predict_proba(X_scaled)[0]
        classes = model.classes_

        predicted_class = classes[np.argmax(probs)]
        confidence = float(round(np.max(probs) * 100, 2))
        exoplanet_classes = ["CONFIRMED", "CANDIDATE"]
        is_exoplanet = predicted_class in exoplanet_classes

        result_msg = (
            f"This is an exoplanet with {confidence}% confidence."
            if is_exoplanet
            else f"This is not an exoplanet with {confidence}% confidence."
        )

        return {
            "Prediction": "Exoplanet" if is_exoplanet else "Not Exoplanet",
            "Confidence (%)": confidence,
            "Message": result_msg
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {str(e)}")


#scientific predict
@app.post("/predict-file/")
def upload_csv(file: UploadFile = File(...)):
    try:
        # ✅ FIX #7: Add unique filename to avoid overwriting
        unique_name = f"{uuid.uuid4()}_{file.filename}"
        temp_path = os.path.join(SCIENTIFIC_OUTPUT_DIR, unique_name)
        with open(temp_path, "wb") as f:
            f.write(file.file.read())

        df = smart_read(temp_path)

        # Mapping
        df_mapped = pd.DataFrame()
        for user_feat, dataset_feat in features.items():
            if user_feat in df.columns:
                df_mapped[dataset_feat] = df[user_feat]
            else:
                df_mapped[dataset_feat] = defaults.get(user_feat, 0)

        # Preprocessing
        df_cleaned = clean_column_names(df_mapped)
        df_no_nulls = drop_high_nulls(df_cleaned)
        df_target = detect_target(df_no_nulls)
        df_engineered = feature_engineering(df_target)
        df_imputed = impute_missing(df_engineered)
        df_encoded = encoding(df_imputed)
        df_outliers_handled = outlier_handling(df_encoded)
        df_scaled = scaling(df_outliers_handled)
        X_scaled = df_scaled[final_features]

        # Model
        preds = model.predict(X_scaled)
        probs = model.predict_proba(X_scaled)
        max_probs = probs.max(axis=1) * 100

        df_mapped["prediction"] = preds
        df_mapped["probability"] = max_probs

        confirmed_df = df_mapped[df_mapped["prediction"] == "CONFIRMED"].copy()

        required_columns = [
            "transit_depth", "transit_duration", "transit_epoch", "orbital_period",
            "planet_to_star_radius_ratio", "planet_radius", "stellar_radius",
            "temperature", "stellar_temperature", "distance", "prediction", "probability",
        ]

        # ✅ FIX #8: Log missing columns instead of silently filling 0
        missing_cols = [col for col in required_columns if col not in confirmed_df.columns]
        if missing_cols:
            print(f"[WARNING] Missing columns in confirmed_df: {missing_cols}")

        for col in required_columns:
            if col not in confirmed_df.columns:
                if col in fill_values:
                    confirmed_df[col] = fill_values[col]
                elif col in defaults:
                    confirmed_df[col] = defaults[col]
                else:
                    confirmed_df[col] = np.nan  # safer than 0

        confirmed_df = confirmed_df[required_columns]

        output_file = os.path.join(SCIENTIFIC_OUTPUT_DIR, f"confirmed_exoplanets_{unique_name}")
        confirmed_df.to_csv(output_file, index=False)

        return FileResponse(
            output_file,
            filename=f"confirmed_exoplanets_{unique_name}",
            media_type="text/csv",
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"CSV Processing Error: {str(e)}")


#visualization
class Planet(BaseModel):
    planet_row_id: int
    orbit: int
    speed: float
    temperature: float
    volume: float
    diameter: float
    flux: float
    distance_from_star: float
    distance_from_earth: float
    discovery_year: int


planets_data = {}

# ✅ FIX #9: Safe numeric conversion
def safe_float(x):
    try:
        return float(x)
    except:
        return 0.0

def safe_int(x):
    try:
        return int(float(x))
    except:
        return 0


try:
    df_existing = pd.read_csv("")  # left intentionally incomplete
    for idx, row in df_existing.iterrows():
        planet = Planet(
            planet_row_id=safe_int(row.get("id", 0)),
            orbit=safe_int(row.get("orbit", 0)),
            speed=safe_float(row.get("speed", 0)),
            temperature=safe_float(row.get("temperature", 0)),
            volume=safe_float(row.get("volume", 0)),
            diameter=safe_float(row.get("diameter", 0)),
            flux=safe_float(row.get("flux", 0)),
            distance_from_star=safe_float(row.get("distance_from_star", 0)),
            distance_from_earth=safe_float(row.get("distance_from_earth", 0)),
            discovery_year=safe_int(row.get("discovery_year", 0)),
        )
        planets_data[planet.planet_row_id] = planet
except FileNotFoundError:
    print("[WARNING] exoplanets.csv not found, planets_data will be empty.")


@app.get("/visualize-planets/")
def visualize_planets():
    return [planet.dict() for planet in planets_data.values()]


[WARNING] exoplanets.csv not found, planets_data will be empty.
